# 17 — Train one member of the two-model diversity experiment

Run this notebook twice on an **A100/H100 80GB**, first with `MODEL_INDEX=0`, then with `1`.
Both runs use one shared manifest: every LIBERO task is retained, while demonstrations are
independently sampled with replacement within task. This is a full-model fine-tune by default.

The start point is the agreed raw `lerobot/pi05_base`. Neither member inherits the common
LIBERO-finetuned solution; their LIBERO specialization comes only from their independently
bootstrapped demonstration multisets. Final weights are pushed to your Hugging Face account.
The shared split manifest is stored in Drive so both runs use exactly the same experiment.

## 1. Install the exact pinned training stack

In [ ]:
EXTRAS = 'train'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration

In [ ]:
from pathlib import Path
from huggingface_hub import HfApi
from google.colab import drive

drive.mount("/content/drive")

MODEL_INDEX = 0                 # rerun a separate copy with 1
STEPS = 3000                    # signal pilot; increase only after both models train cleanly
BATCH_SIZE = 16                 # safe starting point on an 80GB A100/H100
SAVE_FREQ = 500                 # resumable checkpoints within the current Colab runtime
FULL_FINETUNE = True            # planned experiment; False is explicit expert-only fallback
COMPILE_MODEL = False           # avoids a long first-step compile during the pilot
WANDB = False
RESUME = False

HF_USER = HfApi().whoami()["name"]
MODEL_REPOS = [f"{HF_USER}/pi05-base-to-libero-bootstrap-m0-v1",
               f"{HF_USER}/pi05-base-to-libero-bootstrap-m1-v1"]
PERSISTENT_ROOT = Path("/content/drive/MyDrive/pnp_diversity")
MANIFEST_PATH = PERSISTENT_ROOT / "bootstrap_manifest.json"
OUTPUT_DIR = Path(f"/content/pi05_diversity/train_m{MODEL_INDEX}")
print({"member": MODEL_INDEX, "model_repo": MODEL_REPOS[MODEL_INDEX],
       "output": str(OUTPUT_DIR), "full_finetune": FULL_FINETUNE})

## 3. Build or load the shared episode-bootstrap manifest

The file in Drive contains both independently sampled members. Member 1 must load this exact file;
it must not generate a second manifest.

In [ ]:
from pnp.diversity import (bootstrap_manifest_summary,
    build_bootstrap_manifest_from_lerobot, load_bootstrap_manifest,
    save_bootstrap_manifest)

if MANIFEST_PATH.exists():
    manifest = load_bootstrap_manifest(MANIFEST_PATH)
else:
    manifest = build_bootstrap_manifest_from_lerobot()
    save_bootstrap_manifest(manifest, MANIFEST_PATH)
assert manifest["source_model"] == "lerobot/pi05_base", manifest["source_model"]
display(bootstrap_manifest_summary(manifest))
print("manifest:", MANIFEST_PATH)
print("manifest hash:", manifest["manifest_hash"])
print("dataset revision:", manifest["dataset_revision"])
print("raw model revision:", manifest["source_model_revision"])
print("tasks:", manifest["n_tasks"], "source episodes:", manifest["n_source_episodes"])

## 4. Launch training

In [ ]:
import subprocess, sys

args = [sys.executable, str(package_dir / "scripts" / "train_pi05_bootstrap.py"),
        "--manifest", str(MANIFEST_PATH), "--member", str(MODEL_INDEX),
        "--output-dir", str(OUTPUT_DIR),
        "--policy-repo-id", MODEL_REPOS[MODEL_INDEX],
        "--steps", str(STEPS), "--batch-size", str(BATCH_SIZE),
        "--save-freq", str(SAVE_FREQ)]
if not FULL_FINETUNE: args.append("--expert-only")
if COMPILE_MODEL: args.append("--compile-model")
if WANDB: args.append("--wandb")
if RESUME: args.append("--resume")
print("starting member", MODEL_INDEX)
subprocess.run(args, check=True)

## 5. Record the immutable identifiers

In [ ]:
print({"member": MODEL_INDEX, "model_repo": MODEL_REPOS[MODEL_INDEX],
       "manifest_hash": manifest["manifest_hash"], "source_model": manifest["source_model"],
       "steps": STEPS, "batch_size": BATCH_SIZE, "full_finetune": FULL_FINETUNE})
print("Before training member 1, preserve and reuse:", MANIFEST_PATH)